# Native generalized quantum signal processing

Retain the source polynomial 0.99 cos³(x), seven fractional input bits, Laurent shift three, phase finding and 100000 finite samples. This workflow recalculates angles from the polynomial using bounded spectral completion and degree stripping; it never uses frozen phase rows as an answer or fallback.


In [ ]:
import numpy as np

from examples.synthesis.algorithms.quantum_primitives.gqsp.overview.main import (
    plot_results,
    read_workflow,
)
from examples.synthesis.algorithms.quantum_primitives.gqsp.overview.phases import (
    factor_phases,
)

coefficients = 0.99 / 8 * np.array([1.0, 0, 3, 0, 3, 0, 1])
fit = factor_phases(coefficients)
print("New native phase rows:", fit.phases)


## Native quantum program

A seven-bit unsigned fractional x register is initialized uniformly. The source phase generator is expressed as coherent phase arithmetic. Three positive and three inverse signal calls surround the seven single-indicator rotations. The compiler owns wires and reversible work.


In [ ]:
from math import pi
from typing import Annotated

from examples.synthesis.algorithms.quantum_primitives.gqsp.operations import rotate
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QBit,
    QNum,
    X,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def gqsp_program(
    negative_power: int,
    initial_indicator: int,
    theta: list[float],
    phi: list[float],
    lam: list[float],
    x: Output[Annotated[QNum, (7, False, 7)]],
    ind: Output[QBit],
) -> None:
    allocate(ind)
    allocate(x)
    hadamard_transform(x)
    if initial_indicator == 1:
        X(ind)
    rotate(theta[0], phi[0], lam[0], ind)
    for i in range(1, 7 - negative_power):
        phase(x * (1 - ind), 2 * pi)
        rotate(theta[i], phi[i], lam[i], ind)
    for i in range(7 - negative_power, 7):
        phase(x * ind, -2 * pi)
        rotate(theta[i], phi[i], lam[i], ind)


program = synth.synthesize(
    gqsp_program,
    classical_inputs={
        "negative_power": 3,
        "initial_indicator": 0,
        "theta": [row[0] for row in fit.phases],
        "phi": [row[1] for row in fit.phases],
        "lam": [row[2] for row in fit.phases],
    },
    max_workspace=0,
)
print(program.report["resources"])


## Actual execution and finite sampling

Read fractional values with the existing `program.numeric_probabilities` API. No physical wires, integer scaling or bitstring decoding are needed. Run 100000 local multinomial draws from the actual exact native distribution with seed 123. This is local classical sampling, not a hardware run or a new runtime shot API. Check exact polynomial probabilities separately from finite-sample error and retain zero-count bins.


In [ ]:
workflow = read_workflow(program, fit)
assert sum(count for _, count in workflow.counts) == 100000
print("Fractional grid:", workflow.points.tolist())
print("Scaled sampled probabilities:", workflow.measured.tolist())


## Source comparison plot

Postselect on indicator zero and divide the joint probabilities by 0.99²; do not renormalize the selected distribution. Compare to cos⁶(2πx)/128. The plot uses all 128 bins. Different phase solutions can preserve projected probabilities while changing complementary amplitudes; full-state comparison uses identical newly calculated phases. This concrete workflow does not certify general phase-solver realizability or analytic uniform error.

The curve label includes the actual 2πx argument; the source label omits 2π. The legend is placed above the data to keep every bin visible.


In [ ]:
figure = plot_results(workflow)
